# Чистка треков Suno от «цифрового дождя»

Нейросеть **Mel-RoFormer Denoise** (открытая, бесплатная) убирает шелест и «вьюгу» вокруг нот.

**Перед запуском:** меню **Среда выполнения → Сменить среду выполнения → T4 GPU → Сохранить**.

Потом заполните настройки в шаге 1 и нажмите **Среда выполнения → Выполнить всё**. Colab попросит доступ к Google Диску — разрешите.


In [ ]:
#@title 1. Настройки
#@markdown Оставьте пустым — возьму самый свежий архив на Диске. Или впишите имя, например: Distant Clarinet.zip
ARCHIVE = ""  #@param {type:"string"}
#@markdown Сила чистки: aggressive — смелее с шумом поверх тарелок; normal — бережнее.
STRENGTH = "aggressive"  #@param ["aggressive", "normal"]
#@markdown Скорость: fast — примерно вдвое быстрее (на слух почти без разницы); quality — медленнее, максимум качества.
SPEED = "fast"  #@param ["fast", "quality"]


In [ ]:
#@title 2. Подключаем Диск и ставим нейросеть (около минуты)
from google.colab import drive
drive.mount('/content/drive')
!pip -q install "audio-separator[gpu]" > /dev/null
import torch
print('Видеокарта:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'НЕТ — включите T4 GPU в настройках среды!')


In [ ]:
#@title 3. Чистим каждый трек
import os, zipfile, shutil, glob
from audio_separator.separator import Separator

MODELS = {
    'aggressive': 'denoise_mel_band_roformer_aufr33_aggr_sdr_27.9768.ckpt',
    'normal': 'denoise_mel_band_roformer_aufr33_sdr_27.9959.ckpt',
}
if ARCHIVE.strip():
    found = glob.glob('/content/drive/MyDrive/**/' + glob.escape(ARCHIVE.strip()), recursive=True)
    assert found, f'Не нашла на Диске архив «{ARCHIVE}» — проверьте имя'
    src = found[0]
else:
    # самый свежий архив на Диске, кроме уже почищенных
    zips = [p for p in glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True) if '(clean)' not in p]
    assert zips, 'На Диске не нашлось ни одного архива .zip'
    src = max(zips, key=os.path.getmtime)
print('Беру архив:', src)
work = '/content/work'; shutil.rmtree(work, ignore_errors=True)
os.makedirs(f'{work}/in'); os.makedirs(f'{work}/out')

with zipfile.ZipFile(src) as z:
    # порядок файлов в архиве важен сборщику — сохраняем его
    order = [n for n in z.namelist() if n.lower().endswith(('.wav', '.mp3', '.flac')) and not n.startswith('__MACOSX')]
    z.extractall(f'{work}/in')
print('Треков:', len(order))

sep = Separator(output_dir=f'{work}/out', output_format='WAV', model_file_dir='/content/models',
                mdxc_params={'segment_size': 256, 'overlap': 2 if SPEED == 'fast' else 4, 'batch_size': 1})
sep.load_model(model_filename=MODELS[STRENGTH])

clean = {}
for i, name in enumerate(order, 1):
    path = os.path.join(f'{work}/in', name)
    outs = sep.separate(path)
    dry = [o for o in outs if '(dry)' in o or '(Dry)' in o][0]
    clean[name] = os.path.join(f'{work}/out', os.path.basename(dry))
    for o in outs:
        if o != dry:
            try: os.remove(os.path.join(f'{work}/out', os.path.basename(o)))
            except FileNotFoundError: pass
    print(f'{i}/{len(order)} готов: {os.path.basename(name)}')


In [ ]:
#@title 4. Кладём чистый архив рядом с исходным
base, _ = os.path.splitext(src)
dst = base + ' (clean).zip'
with zipfile.ZipFile(dst, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in order:  # те же имена и тот же порядок, что в исходном архиве
        z.write(clean[name], arcname=os.path.basename(name).rsplit('.', 1)[0] + '.wav')
print('Готово:', dst)
print('Дайте на этот архив доступ по ссылке и отправьте ссылку боту, как обычно.')
